# 04 What predicts retrieval success, and which label-free signals work?

This notebook is self-contained. It downloads NFCorpus from the Hugging Face Hub, encodes the corpus, and calculates all numbers and figures. It does not read `results/`, `cache/`, or a `silent_rag` module. You need only the pip packages in the next cell. On a laptop CPU, the run time is approximately 15 to 30 minutes. Most of this time is for the encoder (approximately 3,600 documents) and the bootstrap and cross-validation loops.

**Questions**
1. Which query, document, and retrieval-score features go with nDCG@10 for each query?
2. Which of these features stay significant after multiplicity correction and with cluster-aware uncertainty?
3. Can label-free features predict success on topics that the model did not see?
4. Is the "hard topic cluster" real?

**Method**
- Spearman correlations with a cluster bootstrap (design effect).
- Holm correction for the pre-specified hypotheses H1 and H2. Benjamini-Hochberg correction for the other features.
- VIF and partial correlations.
- Cross-validation with held-out topics, and a permutation null.
- Cluster stability, and a held-out test of the lowest cluster.

**Scope.** This notebook uses only NFCorpus. NFCorpus is the development set of the project. SciFact and FiQA are held-out sets for the final run. This notebook does not load them. H3 (window-level tracking of the top-1 score) is not in this notebook. Thus, the confirmatory Holm family here is only H1 and H2. This notebook does not examine cluster stability across models.

In [ ]:
# On a new environment, remove the "#" from the next line.
# %pip install -q datasets==5.1.0 sentence-transformers==6.1.0 torch==2.14.1 scikit-learn==1.9.1 scipy==1.18.1 pandas==3.0.6 numpy==2.5.4 matplotlib==3.11.2

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # On macOS, this prevents an OpenMP conflict. It has no effect on other systems.
os.environ.setdefault("OMP_NUM_THREADS", "1")

import platform
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from scipy import stats
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import adjusted_rand_score, r2_score, roc_auc_score, silhouette_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200, "display.max_columns", 30)

SEED = 0
np.random.seed(SEED)
K = 10
SPLITS = ["train", "validation", "test"]   # The Hub name of the qrels split is "validation", not "dev".
GROUP_K = 50      # Primary query-group definition (pre-specified). k = 10 and k = 150 are sensitivity checks.
B_BOOT = 1000
MAX_SEQ = 512

NFCORPUS = dict(
    repo="BeIR/nfcorpus", revision="b5026a0e96e8a7ac4f95f482a596389289d46269",
    qrels="BeIR/nfcorpus-qrels", qrels_revision="a451b3b26d3ae1358f259c1a3a4dd61fcea35a65",
)
MODEL = dict(repo="sentence-transformers/multi-qa-MiniLM-L6-cos-v1", revision="b207367332321f8e44f96e224ef15bc607f4dbf0")

## Helpers, data, and clean retrieval

This section defines the functions that load the data, encode the text, do an exact cosine search (top 100), and calculate nDCG@10. nDCG@10 uses graded gain and a log2 discount, as in BEIR. The notebook uses all labelled queries that have one or more relevant documents, from the train, validation, and test qrels. If a query is in more than one split, the notebook keeps the first split.

In [ ]:
def doc_text(title, text):
    return f"{(title or '').strip()} {(text or '').strip()}".strip()


def load_nfcorpus():
    c = NFCORPUS
    corpus = load_dataset(c["repo"], "corpus", split="corpus", revision=c["revision"])
    queries = load_dataset(c["repo"], "queries", split="queries", revision=c["revision"])
    qtext = {str(q["_id"]): q["text"] for q in queries}
    split_of, qrels = {}, {}
    for split in SPLITS:
        for row in load_dataset(c["qrels"], split=split, revision=c["qrels_revision"]):
            q = str(row["query-id"])
            if split_of.setdefault(q, split) == split:
                qrels.setdefault(q, {})[str(row["corpus-id"])] = int(row["score"])
    doc_ids = [str(d["_id"]) for d in corpus]
    docs = [doc_text(t, x) for t, x in zip(corpus["title"], corpus["text"])]
    return doc_ids, docs, {q: qtext[q] for q in qrels}, qrels, split_of


def ndcg_at_k(ranked, rels, k=K):
    dcg = sum(rels.get(d, 0) / np.log2(i + 2) for i, d in enumerate(ranked[:k]))
    ideal = sorted(rels.values(), reverse=True)[:k]
    idcg = sum(r / np.log2(i + 2) for i, r in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def encode(enc, texts, batch_size=32):
    return np.ascontiguousarray(enc.encode(list(texts), batch_size=batch_size, normalize_embeddings=True,
                                           convert_to_numpy=True, show_progress_bar=False), dtype=np.float32)


def search(D, Q, depth=100):
    S = Q @ D.T
    # Stable full sort: if two scores are equal (the corpus has duplicate documents), the lower document index comes first.
    # Thus, the top-k does not depend on the search routine.
    top = np.argsort(-S, axis=1, kind="stable")[:, :depth]
    return top, np.take_along_axis(S, top, axis=1)


doc_ids, docs, queries, qrels, split_of = load_nfcorpus()
enc = SentenceTransformer(MODEL["repo"], revision=MODEL["revision"], device="cpu")
enc.max_seq_length = MAX_SEQ

qids = [q for q in queries if any(r > 0 for r in qrels[q].values())]
split = np.array([split_of[q] for q in qids])
rel = [{d for d, v in qrels[q].items() if v > 0} for q in qids]
D = encode(enc, docs)
Q = encode(enc, [queries[q] for q in qids])
top, scores = search(D, Q, depth=100)
ids = [[doc_ids[j] for j in row] for row in top]
ndcg = np.array([ndcg_at_k(r, qrels[q]) for r, q in zip(ids, qids)])
recall = np.array([len(r & set(i[:K])) / len(r) for r, i in zip(rel, ids)])
print(len(qids), "queries | mean nDCG@10", round(float(ndcg.mean()), 4), "| share of zeros", round(float((ndcg == 0).mean()), 3))

## Features

| Feature | Label-free | Notes |
|---|---|---|
| `q_tokens` | yes | query length |
| `top1_score`, `top10_mean_score`, `top1_top2_gap` | yes | retrieval scores |
| `nn_query_sim`, `q_density` | yes | cosine to the nearest other query, and mean cosine to the 10 nearest other queries, over all labelled queries (transductive; calculated again in each training fold for the CV section) |
| `n_rel` | no | number of relevant documents |
| `rel_doc_tokens`, `rel_frac_over_256` | no | length of the relevant documents |
| `rel_cos` | no | cosine of the query to its relevant documents (near the outcome by construction) |
| `rel_hubness_loo` | no | mean k-occurrence of the relevant documents, **without the top-10 contribution of the query itself** |

`rel_hubness_orig` includes the top-10 contribution of the query itself. The notebook uses it only to measure the circularity.

"Label-free" means that the feature uses no qrels. But the query set is "labelled queries with one or more relevant documents". A production system does not make this selection.

In [ ]:
FEATURES = {"q_tokens": True, "top1_score": True, "top10_mean_score": True, "top1_top2_gap": True, "nn_query_sim": True, "q_density": True,
            "n_rel": False, "rel_doc_tokens": False, "rel_frac_over_256": False, "rel_cos": False, "rel_hubness_loo": False}
LABEL_FREE = [f for f, v in FEATURES.items() if v]

pos = {d: i for i, d in enumerate(doc_ids)}
tok = enc.tokenizer
kocc = np.bincount(top[:, :K].ravel(), minlength=len(D))
S = Q @ Q.T
np.fill_diagonal(S, -np.inf)
nn_top = -np.sort(-S, axis=1)[:, :K]
rel_docs = sorted({d for r in rel for d in r if d in pos})
doc_len = {d: len(tok.tokenize(docs[pos[d]])) for d in rel_docs}

rows = []
for i, r in enumerate(rel):
    r = [d for d in r if d in pos]
    own = set(top[i, :K])
    lens = np.array([doc_len[d] for d in r])
    rows.append({
        "q_tokens": len(tok.tokenize(queries[qids[i]])), "top1_score": scores[i, 0], "top10_mean_score": scores[i, :K].mean(),
        "top1_top2_gap": scores[i, 0] - scores[i, 1], "nn_query_sim": nn_top[i, 0], "q_density": nn_top[i].mean(),
        "n_rel": len(r), "rel_doc_tokens": lens.mean(), "rel_frac_over_256": (lens > 256).mean(),
        "rel_cos": float((D[[pos[d] for d in r]] @ Q[i]).mean()),
        "rel_hubness_orig": float(np.mean([kocc[pos[d]] for d in r])),
        "rel_hubness_loo": float(np.mean([kocc[pos[d]] - (pos[d] in own) for d in r])),
    })
df = pd.DataFrame(rows)
df["ndcg10"] = ndcg
df["failed"] = (recall == 0).astype(int)
df.describe().T.round(3)

## Query groups and the design effect

Queries on the same topic, or queries that share relevant documents, are not independent. Thus, the notebook calculates the uncertainty with a resample of **groups** of queries. There are three group definitions:
- KMeans on the query embeddings, with k = 10, 50 (primary), and 150
- connected components of queries that share a relevant document

The group sizes show how strong the dependence is. If one component holds most queries, shared documents link most queries, and that definition is not usable.

In [ ]:
def components_by_shared_docs(rel_sets):
    parent = list(range(len(rel_sets)))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    owner = {}
    for i, r in enumerate(rel_sets):
        for d in r:
            if d in owner:
                parent[find(i)] = find(owner[d])
            else:
                owner[d] = i
    return np.array([find(i) for i in range(len(rel_sets))])


groups = {f"kmeans{k}": KMeans(k, n_init=5, random_state=SEED).fit_predict(Q) for k in (10, GROUP_K, 150)}
groups["shared_doc_components"] = components_by_shared_docs(rel)
rows = []
for gname, lab in groups.items():
    sizes = pd.Series(lab).value_counts()
    rows.append({"grouping": gname, "n_groups": len(sizes), "largest_group_share": sizes.iloc[0] / len(lab), "median_size": sizes.median()})
pd.DataFrame(rows).round(3)

## Univariate associations with cluster bootstrap

This section calculates the Spearman rho between each feature and nDCG@10. The CI resamples full groups (primary: KMeans k = 50) `B = 1000` times. The p-value is the two-sided bootstrap p, `(1 + 2 * min(#{rho* <= 0}, #{rho* >= 0})) / (B + 1)`. The plain Spearman p (which assumes independence) is also shown for comparison. The **design effect** is the cluster-CI width divided by the iid-CI width. A value above 1 shows that the CI for independent queries is too narrow. If the Spearman of a bootstrap replicate is undefined (a constant resample), the notebook removes the replicate. The notebook counts and reports the removed replicates.

**Why this test**
- **Estimand:** the population Spearman correlation between a feature and nDCG@10 for each query, over NFCorpus queries with this model.
- **Unit of independence:** the KMeans(50) topic group, not the query. Queries on one topic share relevant documents, so a resample of queries gives a variance that is too small. Groups are a proxy for the unknown source page. Thus, these CIs are lower limits (see the sensitivity table in a later section).
- **Assumptions:** groups are approximately exchangeable and independent of each other. With 50 groups, the percentile bootstrap is sufficient. The grouping-sensitivity and KMeans-seed tables check this.
- **Why not the usual alternative:** the analytic Spearman p (`p_plain`) assumes independent queries. nDCG is bounded, zero-inflated (33% zeros), and has ties. Thus, a Fisher-z interval is not reliable. A cluster bootstrap needs no distributional form.
- **Resolution / Monte Carlo error:** the lowest possible bootstrap p-value is `1/(B+1) = 0.001` for B = 1000. The notebook reports a p at this limit as `p < 0.001`, not as an exact value.

In [ ]:
def spearman(x, y):
    return stats.spearmanr(x, y)[0]


NAN_DROPPED = {"total": 0}   # The notebook counts undefined bootstrap replicates. It does not remove them silently.


def boot_rho(x, y, labels, B=B_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    if labels is None:
        n = len(x)
        out = [spearman(x[i], y[i]) for i in (rng.integers(0, n, n) for _ in range(B))]
    else:
        uniq, inv = np.unique(labels, return_inverse=True)
        members = [np.where(inv == g)[0] for g in range(len(uniq))]
        out = []
        for _ in range(B):
            idx = np.concatenate([members[g] for g in rng.integers(0, len(uniq), len(uniq))])
            out.append(spearman(x[idx], y[idx]))
    out = np.array(out)
    boot_rho.last_dropped = int(np.isnan(out).sum())
    NAN_DROPPED["total"] += boot_rho.last_dropped
    return out[~np.isnan(out)]


def boot_p(boot):
    """Return the two-sided bootstrap p for rho = 0: (1 + 2 * smaller tail count) / (B + 1)."""
    return min(1.0, (1 + 2 * min((boot <= 0).sum(), (boot >= 0).sum())) / (len(boot) + 1))


def boot_p_pos(boot):
    """Return the one-sided bootstrap p for rho > 0: (1 + #{rho* <= 0}) / (B + 1)."""
    return (1 + (boot <= 0).sum()) / (len(boot) + 1)


y = df["ndcg10"].to_numpy()
lab = groups[f"kmeans{GROUP_K}"]
rows = []
for feat in list(FEATURES) + ["rel_hubness_orig"]:
    x = df[feat].to_numpy(float)
    if np.ptp(x) == 0:
        continue
    rho, p_plain = stats.spearmanr(x, y)
    b_iid = boot_rho(x, y, None)
    drop_iid = boot_rho.last_dropped
    b_cl = boot_rho(x, y, lab)
    drop_cl = boot_rho.last_dropped
    ci_iid, ci_cl = np.percentile(b_iid, [2.5, 97.5]), np.percentile(b_cl, [2.5, 97.5])
    rows.append({"feature": feat, "label_free": FEATURES.get(feat, False), "n": len(y), "rho": rho,
                 "ci_lo_iid": ci_iid[0], "ci_hi_iid": ci_iid[1], "ci_lo_cluster": ci_cl[0], "ci_hi_cluster": ci_cl[1],
                 "design_effect": (ci_cl[1] - ci_cl[0]) / (ci_iid[1] - ci_iid[0]), "p_plain": p_plain, "p_cluster": boot_p(b_cl),
                 "p_cluster_onesided_pos": boot_p_pos(b_cl), "nan_dropped_iid": drop_iid, "nan_dropped_cluster": drop_cl})
uni = pd.DataFrame(rows)
print("design effect (cluster CI width / iid CI width): median", round(float(uni["design_effect"].median()), 2), "max", round(float(uni["design_effect"].max()), 2))
print("undefined (NaN) bootstrap replicates dropped in this table:", int(uni[["nan_dropped_iid", "nan_dropped_cluster"]].to_numpy().sum()),
      "of", 2 * B_BOOT * len(uni))
uni.round(4)

## Multiplicity

**Confirmatory (Holm):** H1 (the rho of the top-1 score with nDCG@10 is positive) and H2 (the rho of the gap between the top-1 and top-2 scores is positive). The tests use the one-sided cluster-bootstrap p `(1 + #{rho* <= 0}) / (B + 1)`. The lowest possible bootstrap p is 1/(B+1) ≈ 0.001. Thus, the Holm-adjusted values cannot be less than approximately 0.002. A value at this limit means `p < 0.002`, not an exact p.

**Exploratory (Benjamini-Hochberg, FDR 5%):** all two-sided cluster-bootstrap p-values, except `rel_hubness_orig`.

**Why this test**
- **Estimand:** for H1 and H2, the sign of rho for the two signals that the analysis plan of the project names as hypotheses. The plan is not under version control, so it is not possible to show that it was written before these analyses. For the exploratory rows, the estimand is which other associations are different from zero.
- **Unit of independence:** the KMeans(50) group, from the bootstrap in the previous section.
- **Assumptions:** Holm controls the family-wise error rate under all dependence between the two tests (top-1 and gap are correlated). BH controls FDR under positive dependence. This is plausible for correlated score features, but it is not guaranteed.
- **Why not the usual alternative:** Bonferroni has the same guarantee as Holm and less power. All 11 features in the confirmatory family would decrease the power for the two hypotheses. H1 is almost mechanical, because top-1 and nDCG come from the same retrieval. Thus, the *size* of rho is the important result, not the rejection.
- **Resolution / Monte Carlo error:** the lowest possible value for each test is `1/(B+1)`. At this limit, the Holm value for the family of two is approximately 0.002.

In [ ]:
def holm(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    out, run = np.empty(len(p)), 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, p[i] * (len(p) - rank)))
        out[i] = run
    return out


def bh(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    ranked = p[order] * len(p) / (np.arange(len(p)) + 1)
    adj = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(len(p))
    out[order] = np.minimum(adj, 1.0)
    return out


conf = uni[uni["feature"].isin(["top1_score", "top1_top2_gap"])][["feature", "rho", "ci_lo_cluster", "ci_hi_cluster", "p_cluster_onesided_pos"]].copy()
conf["hypothesis"] = conf["feature"].map({"top1_score": "H1", "top1_top2_gap": "H2"})
conf["p_holm"] = holm(conf["p_cluster_onesided_pos"])
conf["reject_at_0.05"] = conf["p_holm"] < 0.05
display(conf.round(4))

exp = uni[uni["feature"] != "rel_hubness_orig"].copy()
exp["p_bh"] = bh(exp["p_cluster"])
exp["significant_fdr05"] = exp["p_bh"] < 0.05
exp[["feature", "label_free", "rho", "ci_lo_cluster", "ci_hi_cluster", "design_effect", "p_bh", "significant_fdr05"]].round(4)

In [ ]:
order_ = exp.sort_values("rho")
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.barh(order_["feature"], order_["rho"], color=np.where(order_["label_free"], "tab:blue", "lightgrey"),
        xerr=[order_["rho"] - order_["ci_lo_cluster"], order_["ci_hi_cluster"] - order_["rho"]], capsize=2)
ax.axvline(0, color="grey")
ax.set_xlabel("Spearman rho with nDCG@10 (95% cluster CI); blue = label-free")
fig.tight_layout()
plt.show()

## Sensitivity of the CIs to the group definition

This section calculates the CIs of the two confirmatory features with each grouping, and with the plain bootstrap. If the result changes with the grouping, the grouping choice is important, and the report must state it.

**Why this test**
- **Estimand:** how much the interval for rho depends on the assumed unit of independence.
- **Unit of independence:** this section changes it on purpose: query (iid), KMeans k = 150, 50, and 10, and shared-document components.
- **Assumptions:** only the assumptions of the bootstrap. If the interval becomes wider as the groups become larger, the dependence extends beyond the smaller groups.
- **Why not the usual alternative:** without source-page metadata, it is not possible to justify one "best" grouping. Thus, the range is the correct summary. The interval for the widest *usable* grouping (k = 10) is the conservative value to report.
- **Resolution / Monte Carlo error:** 500 replicates for each row. The Monte Carlo SE of a 2.5% percentile endpoint is approximately 0.005 in rho.

In [ ]:
rows = []
for feat in ("top1_score", "top1_top2_gap"):
    x = df[feat].to_numpy(float)
    rho = spearman(x, y)
    for gname, lab_ in [("iid", None)] + list(groups.items()):
        b = boot_rho(x, y, lab_, B=500)
        lo, hi = np.percentile(b, [2.5, 97.5])
        rows.append({"feature": feat, "grouping": gname, "rho": rho, "ci_lo": lo, "ci_hi": hi, "width": hi - lo, "nan_dropped": boot_rho.last_dropped})
pd.DataFrame(rows).round(3)

## Stability of the topic grouping (KMeans seed)

The primary groups use KMeans seed 0. This section calculates the confirmatory CIs again with five different seeds. The spread is the part of the uncertainty that comes from the grouping.

In [ ]:
rows = []
for gs in range(5):
    lab_s = KMeans(GROUP_K, n_init=5, random_state=gs).fit_predict(Q)
    for feat in ("top1_score", "top1_top2_gap"):
        x = df[feat].to_numpy(float)
        b = boot_rho(x, y, lab_s, B=500)
        lo, hi = np.percentile(b, [2.5, 97.5])
        rows.append({"grouping_seed": gs, "feature": feat, "rho": spearman(x, y), "ci_lo": lo, "ci_hi": hi,
                     "design_effect": (hi - lo) / np.ptp(np.percentile(boot_rho(x, y, None, B=500), [2.5, 97.5]))})
t = pd.DataFrame(rows)
t.groupby("feature").agg(rho=("rho", "first"), ci_lo_min=("ci_lo", "min"), ci_lo_max=("ci_lo", "max"), ci_hi_min=("ci_hi", "min"),
                         ci_hi_max=("ci_hi", "max"), design_effect_min=("design_effect", "min"), design_effect_max=("design_effect", "max")).round(3)

## The circularity of `rel_hubness`

`rel_hubness_orig` counts the retrievals of a relevant document, and this count includes the retrievals for the query itself. If the query retrieves its own relevant document, that document adds to its own k-occurrence. Thus, the feature partly includes the outcome. This table compares `rel_hubness_orig` with the leave-one-out version (cluster CIs, from the univariate table):

In [ ]:
uni[uni["feature"].isin(["rel_hubness_orig", "rel_hubness_loo"])][["feature", "rho", "ci_lo_cluster", "ci_hi_cluster"]].round(3)

## Collinearity and partial correlations

Variance inflation factors (VIF = 1 / (1 - R^2) of each feature regressed on the other features, on ranks) measure the collinearity. Collinearity makes regression coefficients difficult to interpret. Partial Spearman correlations (ranks residualized on the control set) separate the score signals from each other:
- `top1_score` given `top1_top2_gap` and `top10_mean_score`
- `top1_top2_gap` given the other two

The CIs are cluster-bootstrap CIs.

**Why this test**
- **Estimand:** the rank association between a signal and nDCG@10 that stays after a linear adjustment (on ranks) for the other signals.
- **Unit of independence:** the KMeans(50) group (cluster bootstrap, 300 replicates).
- **Assumptions:** the confounding is approximately linear in ranks. VIF is descriptive and needs no distributional assumption.
- **Why not the usual alternative:** regression coefficients on collinear features (VIF 9 to 12) are not stable, and their sign can change easily. A partial correlation answers a narrower and more stable question: "Does this signal add information when the other signals are known?"
- **Resolution / Monte Carlo error:** 300 replicates. The CI endpoints have a Monte Carlo SE of approximately 0.01.

In [ ]:
X = df[list(FEATURES)].rank().to_numpy()
X = (X - X.mean(0)) / (X.std(0) + 1e-12)
vif = []
for j, f in enumerate(FEATURES):
    others = np.delete(X, j, axis=1)
    r2 = LinearRegression().fit(others, X[:, j]).score(others, X[:, j])
    vif.append({"feature": f, "VIF": 1 / max(1 - r2, 1e-9)})
display(pd.DataFrame(vif).round(1))


def partial_spearman(frame, x, controls, y="ndcg10"):
    R = frame[[x, y] + controls].rank()
    C = np.column_stack([np.ones(len(R)), R[controls].to_numpy()])
    res = lambda v: v - C @ np.linalg.lstsq(C, v, rcond=None)[0]
    return np.corrcoef(res(R[x].to_numpy()), res(R[y].to_numpy()))[0, 1]


uniq, inv = np.unique(groups[f"kmeans{GROUP_K}"], return_inverse=True)
members = [np.where(inv == g)[0] for g in range(len(uniq))]
rng = np.random.default_rng(SEED)
specs = [("top1_score", ["top1_top2_gap", "top10_mean_score"]), ("top1_top2_gap", ["top1_score", "top10_mean_score"]),
         ("top1_score", ["n_rel", "q_tokens"]), ("q_density", ["top1_score", "top1_top2_gap"])]
rows = []
for x_, ctrl in specs:
    est = partial_spearman(df, x_, ctrl)
    boots = [partial_spearman(df.iloc[np.concatenate([members[g] for g in rng.integers(0, len(uniq), len(uniq))])], x_, ctrl) for _ in range(300)]
    rows.append({"feature": x_, "controlling_for": "+".join(ctrl), "partial_rho": est,
                 "ci_lo": np.percentile(boots, 2.5), "ci_hi": np.percentile(boots, 97.5)})
pd.DataFrame(rows).round(3)

## Out-of-sample prediction with topic-held-out folds

Can the features **predict** nDCG@10 for queries on topics that the model did not see? This section uses 5-fold `GroupKFold` with groups from KMeans k = 50, repeated over 5 different group assignments. There are two models: a ridge regression and a gradient-boosted tree model (non-linear, with interactions). Each model uses the **label-free** features and **all** features.

**Two design points.**
1. *The point estimate and the CI describe the same quantity.* The notebook calculates the mean of the out-of-fold predictions of the 5 repeats for each query (`pm`). The table reports `r2_of_mean_pred = R^2(y, pm)` with its group-bootstrap CI. The mean of the 5 R^2 values of the repeats is a separate column (`r2_mean_of_repeats`). It is a different statistic, and it has no CI here.
2. *Fold-internal density features.* The transductive versions of `nn_query_sim` and `q_density` use all queries. Thus, the feature of a held-out query uses the training queries *and* the other held-out queries. In the `fold-internal` rows, the notebook calculates these features again in each fold. The neighbours of a training query come from the other queries of the training fold. The neighbours of a held-out query come only from the queries of the training fold. The `transductive` rows show the transductive version for comparison.

A coarser row holds out KMeans k = 10 groups (fold-internal features), because adjacent k = 50 clusters can share topics across folds. The table reports the R^2 of the averaged out-of-fold prediction, with a 95% group-bootstrap CI (groups of the same k as the folds). It also reports the AUROC of the same averaged prediction for failure (recall@10 = 0). A permutation test (100 shuffles of nDCG across queries; ridge, label-free, fold-internal) gives the p-value for "R^2 > 0".

**Assumption checks.**
1. Linearity: if the gradient-boosted model is better than ridge out of sample, the linear assumption is too strong.
2. nDCG is bounded and zero-inflated. Thus, the notebook plots the residuals of the ridge fit.

**Why this test**
- **Estimand:** the predictive R^2 of label-free features for nDCG@10 on queries from topic groups that are not in the training data.
- **Unit of independence:** the topic group. The folds and the CI both resample full groups.
- **Assumptions:** held-out groups are exchangeable with training groups. The features use no information from the held-out fold. This is also true for the fold-internal density features.
- **Why not the usual alternative:** random K-fold CV puts near-duplicate and same-topic queries on both sides of a split, and this increases R^2 incorrectly. In-sample R^2 measures fit, not prediction.
- **Resolution / Monte Carlo error:** 300 bootstrap replicates for the CI. The lowest possible permutation p is `1/101 ≈ 0.0099`. Thus, a p at this limit means `p < 0.01`.

In [ ]:
def make_model(kind):
    if kind == "ridge":
        return Ridge(alpha=1.0)
    return HistGradientBoostingRegressor(max_depth=3, max_iter=150, learning_rate=0.05, random_state=SEED)


DENS_COLS = ("nn_query_sim", "q_density")
_dens_cache = {}


def fold_density(tr, te):
    """Return nn_query_sim and q_density from training-fold queries only. A training query does not use itself."""
    key = te.tobytes()
    if key not in _dens_cache:
        S_tr = Q[tr] @ Q[tr].T
        np.fill_diagonal(S_tr, -np.inf)
        top_tr = -np.sort(-S_tr, axis=1)[:, :K]
        top_te = -np.sort(-(Q[te] @ Q[tr].T), axis=1)[:, :K]
        _dens_cache[key] = ({"nn_query_sim": top_tr[:, 0], "q_density": top_tr.mean(1)},
                            {"nn_query_sim": top_te[:, 0], "q_density": top_te.mean(1)})
    return _dens_cache[key]


def fold_X(X, feats, tr, te, fold_internal):
    X_tr, X_te = X[tr].copy(), X[te].copy()
    if fold_internal:
        d_tr, d_te = fold_density(tr, te)
        for c in DENS_COLS:
            if c in feats:
                j = feats.index(c)
                X_tr[:, j], X_te[:, j] = d_tr[c], d_te[c]
    return X_tr, X_te


def oof_predict(X, y, groups_, kind, feats=None, fold_internal=False):
    pred = np.empty(len(y))
    for tr, te in GroupKFold(5).split(X, y, groups_):
        X_tr, X_te = fold_X(X, feats or [], tr, te, fold_internal)
        sc = StandardScaler().fit(X_tr)
        pred[te] = make_model(kind).fit(sc.transform(X_tr), y[tr]).predict(sc.transform(X_te))
    return pred


def group_boot(metric, y_, pred, labels, B=300, seed=SEED):
    uniq_, inv_ = np.unique(labels, return_inverse=True)
    mem = [np.where(inv_ == g)[0] for g in range(len(uniq_))]
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(B):
        idx = np.concatenate([mem[g] for g in rng.integers(0, len(uniq_), len(uniq_))])
        try:
            out.append(metric(y_[idx], pred[idx]))
        except ValueError:
            pass
    return np.percentile(out, [2.5, 97.5]), B - len(out)


failed = df["failed"].to_numpy()
_km = {}


def kmeans_rep(k, rep):
    if (k, rep) not in _km:
        _km[(k, rep)] = KMeans(k, n_init=3, random_state=SEED + rep).fit_predict(Q)
    return _km[(k, rep)]


specs = [(fs, kind, var, GROUP_K) for fs in ("label-free", "all") for kind in ("ridge", "gbm") for var in ("transductive", "fold-internal")]
specs += [("label-free", kind, "fold-internal", 10) for kind in ("ridge", "gbm")]
cv_rows, oof_store = [], {}
for fs_name, kind, var, k in specs:
    feats = LABEL_FREE if fs_name == "label-free" else list(FEATURES)
    Xf = df[feats].to_numpy(float)
    r2s, preds = [], []
    for rep in range(5):
        p = oof_predict(Xf, y, kmeans_rep(k, rep), kind, feats, fold_internal=(var == "fold-internal"))
        r2s.append(r2_score(y, p)); preds.append(p)
    pm = np.mean(preds, axis=0)
    (lo, hi), n_fail_boot = group_boot(r2_score, y, pm, kmeans_rep(k, 0))
    oof_store[(fs_name, kind, var, k)] = pm
    cv_rows.append({"features": fs_name, "model": kind, "density_features": var, "fold_groups": f"kmeans{k}",
                    "r2_of_mean_pred": r2_score(y, pm), "r2_ci_lo": lo, "r2_ci_hi": hi,
                    "r2_mean_of_repeats": np.mean(r2s), "r2_sd_over_repeats": np.std(r2s, ddof=1),
                    "auroc_failure_of_mean_pred": roc_auc_score(failed, -pm), "boot_replicates_failed": n_fail_boot})
cv = pd.DataFrame(cv_rows)
cv.round(4)

In [ ]:
# Permutation test: is the out-of-fold R^2 (ridge, label-free, fold-internal density features) above zero?
# The features do not depend on y. Thus, all permutations use the same density cache for each fold.
lab = groups[f"kmeans{GROUP_K}"]
Xlf = df[LABEL_FREE].to_numpy(float)
obs = r2_score(y, oof_predict(Xlf, y, lab, "ridge", LABEL_FREE, fold_internal=True))
rng = np.random.default_rng(SEED)
null = np.array([r2_score(y, oof_predict(Xlf, rng.permutation(y), lab, "ridge", LABEL_FREE, fold_internal=True)) for _ in range(100)])
pd.DataFrame([{"oof_r2": obs, "null_mean": null.mean(), "null_p95": np.percentile(null, 95), "perm_p": (1 + (null >= obs).sum()) / 101}]).round(4)

In [ ]:
# Permutation importance of the label-free features, measured on held-out folds (gbm, fold-internal density features).
# The score features are collinear (VIF 9 to 12). A permutation of one feature keeps its correlated partner.
# Thus, the split of importance between top1_score and top10_mean_score has no clear meaning.
# This cell also shows the R^2 loss when all three score features are removed together.
imp = np.zeros((5, len(LABEL_FREE)))
grp_drop = np.zeros(5)
score_cols = [LABEL_FREE.index(c) for c in ("top1_score", "top10_mean_score", "top1_top2_gap")]
for f, (tr, te) in enumerate(GroupKFold(5).split(Xlf, y, lab)):
    X_tr, X_te = fold_X(Xlf, LABEL_FREE, tr, te, True)
    sc = StandardScaler().fit(X_tr)
    m = make_model("gbm").fit(sc.transform(X_tr), y[tr])
    imp[f] = permutation_importance(m, sc.transform(X_te), y[te], n_repeats=10, random_state=SEED, scoring="r2").importances_mean
    keep = [j for j in range(len(LABEL_FREE)) if j not in score_cols]
    sc2 = StandardScaler().fit(X_tr[:, keep])
    m2 = make_model("gbm").fit(sc2.transform(X_tr[:, keep]), y[tr])
    grp_drop[f] = r2_score(y[te], m.predict(sc.transform(X_te))) - r2_score(y[te], m2.predict(sc2.transform(X_te[:, keep])))
display(pd.DataFrame({"feature": LABEL_FREE, "r2_drop_mean": imp.mean(0), "r2_drop_sd_over_folds": imp.std(0, ddof=1)}).round(4))
print("drop-column importance of the three score features together (held-out R^2 lost):",
      round(float(grp_drop.mean()), 4), "+/-", round(float(grp_drop.std(ddof=1)), 4), "(SD over folds)")

In [ ]:
p = oof_store[("label-free", "ridge", "fold-internal", GROUP_K)]
fig, ax = plt.subplots(figsize=(4, 3))
ax.scatter(p, y - p, s=3, alpha=0.3)
ax.axhline(0, color="grey")
ax.set_title("ridge residuals (out-of-fold, label-free, fold-internal density)", fontsize=8)
ax.set_xlabel("predicted nDCG@10")
ax.set_ylabel("residual")
fig.tight_layout()
plt.show()
print("NaN bootstrap replicates dropped by boot_rho across the whole notebook so far:", NAN_DROPPED["total"])

## Cluster stability across seeds and k

This section uses 20 KMeans seeds for each k on the query embeddings. Stability is the mean **adjusted Rand index (ARI)** of the partition of each seed against the partition of seed 0 (1 = identical, 0 = chance agreement). **Silhouette** (cosine, subsample of 2,000 queries) measures how well the clusters are separated. A low silhouette with a high ARI shows that the partition is reproducible, but the clusters are not sharply separated. This is usual for text embeddings. It means that cluster identity is only an approximate description.

In [ ]:
sub = np.random.default_rng(SEED).choice(len(Q), min(2000, len(Q)), replace=False)
rows = []
for k in (5, 10, 20, 50):
    labs = [KMeans(k, n_init=3, random_state=s).fit_predict(Q) for s in range(20)]
    ari = [adjusted_rand_score(labs[0], l) for l in labs[1:]]
    sil = [silhouette_score(Q[sub], l[sub], metric="cosine") for l in labs[:5]]
    rows.append({"k": k, "ARI_vs_seed0_mean": np.mean(ari), "ARI_min": np.min(ari), "silhouette_mean": np.mean(sil)})
pd.DataFrame(rows).round(3)

## Is the lowest cluster really low?

If you select the worst of k clusters and test it as if you chose it in advance, the significance is too high. This section uses two corrections:

- **Permutation of the minimum.** The statistic is the most negative standardized cluster deviation `min_c (mean_c - mean) / (sd / sqrt(n_c))`. The null shuffles nDCG across queries (2,000 times) and calculates the same minimum. The p-value is the fraction of null minima that are equally or more extreme.
- **Held-out check.** For each of 20 seeds, the notebook selects the lowest cluster with **train queries only**. Then it compares the nDCG of that cluster with the other clusters on the **held-out queries** (validation and test), with a bootstrap CI of the difference. The fraction of seeds with a negative held-out difference, and its median, show if the "hard cluster" replicates on queries that the selection did not use.

**Why this test**
- **Estimand:** whether the lowest topic cluster is lower than the selection alone causes, and whether this is also true on queries that the selection did not use.
- **Unit of independence:** the query, for the permutation of the minimum (it breaks the link between cluster and nDCG). The query in each split, for the held-out check.
- **Assumptions:** under the null, nDCG is exchangeable across clusters. The held-out check assumes that held-out queries are independent of training queries. This is **false** here: they share clusters and relevant documents. Thus, the replication is partly a shared-document effect.
- **Why not the usual alternative:** a t-test on the selected cluster ignores that the cluster is the minimum of k (the winner's curse). The Kruskal-Wallis test is only an omnibus check, and it treats queries as independent.
- **Resolution / Monte Carlo error:** the lowest possible permutation p is `1/2001 ≈ 0.0005`. The held-out CIs come from 1,000 query-bootstrap replicates (iid, so they are too narrow).

In [ ]:
def min_z(nd, labels):
    z = []
    for c in np.unique(labels):
        m = labels == c
        if m.sum() > 5:
            z.append((nd[m].mean() - nd.mean()) / (nd.std(ddof=1) / np.sqrt(m.sum())))
    return min(z)


rows = []
for k in (10, 50):
    lab_k = KMeans(k, n_init=5, random_state=SEED).fit_predict(Q)
    obs_z = min_z(ndcg, lab_k)
    rng = np.random.default_rng(SEED)
    null = np.array([min_z(rng.permutation(ndcg), lab_k) for _ in range(2000)])
    kw = stats.kruskal(*[ndcg[lab_k == c] for c in np.unique(lab_k)])
    rows.append({"k": k, "min_z": obs_z, "perm_p_min": (1 + (null <= obs_z).sum()) / 2001, "kruskal_H": kw.statistic, "kruskal_p": kw.pvalue})
display(pd.DataFrame(rows).round(4))

train, hold = split == "train", split != "train"
diffs = []
for seed in range(20):
    lab_s = KMeans(10, n_init=3, random_state=seed).fit_predict(Q)
    means = {c: ndcg[train & (lab_s == c)].mean() for c in np.unique(lab_s) if (train & (lab_s == c)).sum() >= 10}
    low = min(means, key=means.get)
    inside, outside = ndcg[hold & (lab_s == low)], ndcg[hold & (lab_s != low)]
    if len(inside) < 10:
        continue
    rng = np.random.default_rng(seed)
    b = [inside[rng.integers(0, len(inside), len(inside))].mean() - outside[rng.integers(0, len(outside), len(outside))].mean() for _ in range(1000)]
    diffs.append({"seed": seed, "low_cluster_train_mean": means[low], "heldout_inside": inside.mean(), "heldout_outside": outside.mean(),
                  "heldout_diff": inside.mean() - outside.mean(), "ci_lo": np.percentile(b, 2.5), "ci_hi": np.percentile(b, 97.5), "n_inside": len(inside)})
d = pd.DataFrame(diffs)
pd.DataFrame([{"seeds_used": len(d), "median_heldout_diff": d["heldout_diff"].median(), "share_seeds_negative": (d["heldout_diff"] < 0).mean(),
               "share_seeds_ci_excludes_0_negative": (d["ci_hi"] < 0).mean(), "median_train_mean_of_low_cluster": d["low_cluster_train_mean"].median(),
               "median_n_heldout_inside": d["n_inside"].median()}]).round(4)

In [ ]:
import datasets, matplotlib, sentence_transformers, sklearn, scipy, torch
print({"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "matplotlib": matplotlib.__version__, "scipy": scipy.__version__,
       "sklearn": sklearn.__version__, "torch": torch.__version__, "sentence_transformers": sentence_transformers.__version__,
       "datasets": datasets.__version__, "seed": SEED, "device": "cpu", "max_seq_length": MAX_SEQ, "group_k": GROUP_K, "bootstrap_B": B_BOOT})

## Findings
All numbers come from one full run (seed 0, NFCorpus, multi-qa-MiniLM, 3,237 queries, mean nDCG@10 0.296; 33% of queries have nDCG 0).

- **Top-1 score has the largest label-free correlation with success on the clean system.** Spearman rho is 0.52 (cluster CI 0.47 to 0.57). The top-10 mean score is 0.41 (CI 0.35 to 0.47). These two CIs almost touch, and this notebook does not test the difference. Thus, the data does not show that top-1 is better than the top-10 mean.
- **The top-1 gap is weaker** (0.20, CI 0.16 to 0.24). H1 and H2 are both significant after Holm (adjusted p < 0.002, the resolution limit). H1 is almost mechanical, because top-1 and nDCG come from the same retrieval. Thus, the size of rho is the important part.
- **Query length and query density are weaker.** Query tokens 0.06 (CI -0.004 to 0.11, BH-adjusted p 0.09, not significant). Density 0.27.
- **The score signals are collinear** (VIF 8.6 for top-1, 11.6 for the top-10 mean). When the other two are known, top-1 keeps a partial rho of 0.33. The gap does not (-0.06). With the number of relevant documents and the query length as controls, top-1 keeps 0.48.
- **Label-free prediction is real but modest.** On topic groups that are not in the training data (k = 50 folds, density features calculated in the training fold), the R^2 of the averaged out-of-fold prediction is 0.284 (95% group-bootstrap CI 0.238 to 0.322) for ridge, and 0.311 (0.266 to 0.354) for gradient boosting. The permutation p is < 0.01 (limit 0.0099). The AUROC for failure is approximately 0.80. With coarser k = 10 folds, R^2 is 0.273 (0.203 to 0.325) and 0.298 (0.218 to 0.358). With all features, including the labelled features, R^2 is 0.47 to 0.59. This is near the outcome by construction.
- **The transductive density features did not increase R^2.** When the notebook calculates them in the training folds, the label-free R^2 changes by less than 0.01 (ridge 0.276 → 0.284, gbm 0.317 → 0.311).
- **The score family as a whole carries most of the signal.** When all three score features are removed together, the held-out R^2 decreases by 0.28 ± 0.07. The score features are collinear, so the importance of one score feature alone has no clear meaning.
- **Circularity is small here.** `rel_hubness_orig` and the leave-one-out `rel_hubness` have correlations of 0.025 (CI -0.023 to 0.071) and -0.007 (CI -0.054 to 0.039) with nDCG.
- **The lowest cluster is also lower on held-out queries** (median difference -0.132, negative in 20 of 20 seeds). But the cluster membership is only moderately stable (ARI 0.41 to 0.70, silhouette approximately 0.04 to 0.06).
- **The CI width depends on the grouping.** For top-1: 0.052 (independent), 0.076 (k = 150), 0.102 (k = 50), 0.146 (k = 10). The shared-document grouping is not usable (one component holds 99.3% of queries).
- **No bootstrap replicate was removed** as undefined (0 NaN replicates in the notebook).

### What this supports
On clean NFCorpus retrieval with this model, the top-1 similarity score of a query is a moderate predictor of query difficulty. When top-1 is known, no other single label-free signal adds much information. The out-of-sample estimate stays the same with fold-internal features and with coarser held-out groups.

### What this does not support
- The selection of the detector signal from this notebook alone. These results measure difficulty for each query on a clean system. They do not measure whether top-1 changes when retrieval degrades.
- That top-1 is better than the top-10 mean score. The CIs almost touch, and the difference is not tested.
- That a feature *causes* success or failure. The data is observational, and the number of relevant documents and the query type are confounders.
- That these correlations are the same **under injected failures**. All features come from a clean system. Thus, this is a screening result. It is not evidence that a detector will alert when retrieval degrades.
- **Unbiased detector numbers on these queries.** If the project selects top-1 as the detector signal because of its correlation with nDCG on these 3,237 queries, power and AUROC values on the same queries are in-sample for that selection. Tuning on NFCorpus is permitted, but these values are optimistic until the held-out run.
- That the order of the signals is the same on other datasets or models. 
- That the "hard cluster" is a stable topic with a clear name, or that it generalizes to new topics. Held-out queries share clusters and relevant documents with training queries.
- That the k = 50 CIs show all of the uncertainty. Larger groups give wider intervals. Thus, these intervals are lower limits.
- That the "label-free" features are fully the same as in production. The query set includes only queries with one or more relevant documents.